# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook:** c12_15 — Geometry & Training Data Generation  
**Author:** Jasper Cluistra  
**Last Updated:** 2026-05-17

---

### Goal
Generate the full training dataset for the GNN surrogate model in a single pass. For each sample, a random spatial truss geometry is drawn, a mean-EA FEM solve estimates structural demand per member, and cross-sections are randomly assigned from the stock pool.

### Inputs
- Grid definition and vertex movement constraints (`c00_headquarter_params`)
- Timber stock CSV (`complete_timber_v2.csv`)

### Outputs
| File | Description |
|------|-------------|
| `dataset_{GRID}_{N}_nodes.csv` | Node features: coordinates, BCs, tributary loads — one row per node per sample |
| `dataset_{GRID}_{N}_edges.csv` | Edge features: geometry + stock properties + N_mean_EA — no Utilization yet |
| `search_space_{GRID}.json` | Discrete/continuous design variables for the optimizer |
| `edge_index_{GRID}.json` | Fixed graph topology (shared across all samples) |

> Centroid + PCA normalisation is applied during generation — do not repeat it in Grasshopper.

## 1.1 Corner index verification
Verify that `get_corner_indices` returns the correct support node positions for different grid sizes. Corner nodes are fixed supports; all other top-layer nodes carry load.

In [ ]:
import c00_headquarter_params as c11_params
import config
from c12_geometry_truss import get_corner_indices

# ── Test ───────────────────────────────────────────────────────────────────────
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 grid indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 grid indices: {grid_3x3}")

grid_5x3 = get_corner_indices(5, 3)
print(f"5x3 grid indices: {grid_5x3}")
# Expected: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> because there are 5 points per row

# ── In the loop ───────────────────────────────────────────────────────────────────────
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] for example
print(corner_values)

## 1.2 Dataset generation
Generates geometry and training data in a single pass using `generate_training_samples`. Output CSVs contain all columns needed for Grasshopper geometry reconstruction and GNN training. Adjust `N_SAMPLES` and `RANDOM_SEED` as needed.

In [ ]:
import config
import c00_headquarter_params as c11_params
import c12_geometry_truss as gen

# ── Parameters ────────────────────────────────────────────────────────────────
N_SAMPLES         = c11_params.NUM_SAMPLES
RANDOM_SEED       = 42
VERBOSE           = False
SAVE_INTERMEDIATE = 0   # save checkpoint every N samples (0 = off)

# ── Run ───────────────────────────────────────────────────────────────────────
result = gen.generate_training_samples(
    n_samples         = N_SAMPLES,
    name_prefix       = f"dataset_{c11_params.GRID}_{N_SAMPLES}",
    output_dir        = config.GEOM_DATA_PATH,
    random_seed       = RANDOM_SEED,
    verbose           = VERBOSE,
    save_intermediate = SAVE_INTERMEDIATE,
)

df_nodes = result["df_nodes"]
df_edges = result["df_edges_out"]

print(f"Generated {result['n_generated']} samples  (skipped: {result['n_skipped']})")
print(f"\nNodes CSV: {result['nodes_csv_path'].name}  ({len(df_nodes)} rows)")
print(f"Edges CSV: {result['edges_csv_path'].name}  ({len(df_edges)} rows)")
print(f"\nNode columns:  {list(df_nodes.columns)}")
print(f"Edge columns:  {list(df_edges.columns)}")

### Output inspection
Inspect the CSVs produced in section 1.2. The edges CSV does not yet contain a `Utilization` column — Karamba FEA fills that in via Grasshopper.

**Next step:** load both CSVs in Grasshopper, run the Karamba FEA simulation, export the edges CSV with the `Utilization` column added, then run `c21_surrogate_training.run_preprocessing()`.

In [ ]:
import c12_geometry_truss as gen

print(f"Samples generated:  {result['n_generated']}  (skipped: {result['n_skipped']})")
print(f"\nNode CSV:  {result['nodes_csv_path'].name}  ({len(df_nodes)} rows)")
print(f"Edge CSV:  {result['edges_csv_path'].name}  ({len(df_edges)} rows)")

print(f"\nEdge feature columns (GNN input after Karamba adds Utilization):")
print(f"  {gen.NEW_EDGE_COLS}")

print(f"\nN_mean_EA range:  {df_edges['N_mean_EA'].min():.0f} N  to  {df_edges['N_mean_EA'].max():.0f} N")
print(f"Width_m range:    {df_edges['Width_m'].min()*1000:.0f} mm  to  {df_edges['Width_m'].max()*1000:.0f} mm")
print(f"Depth_m range:    {df_edges['Depth_m'].min()*1000:.0f} mm  to  {df_edges['Depth_m'].max()*1000:.0f} mm")

print(f"\nFirst 3 edge rows (sample 0):")
display(df_edges[df_edges["sample_id"] == 0][gen.NEW_EDGE_COLS + ["edge_id"]].head(3))

print(f"\nFirst 3 node rows (sample 0):")
display(df_nodes[df_nodes["sample_id"] == 0].head(3))

## 1.5 Design domain
Exports two JSON files used downstream by the optimizer and the GNN:

- **`search_space_{GRID}.json`** — discrete and continuous variables defining how each vertex can move. The optimizer uses this to know the valid design domain.
- **`edge_index_{GRID}.json`** — fixed graph topology (V1/V2 pairs) shared by all structures. Encoding topology once rather than per-sample reduces the GNN's input dimensionality and improves generalisation.

In [ ]:
import json
import config
import c00_headquarter_params as c11_params
from c12_geometry_truss import generate_edges, define_search_space

# ── Run and Review ───────────────────────────────────────────────────────────────
# Use the variables from your earlier configuration
search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
df_topology = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)
edge_index = df_topology[['V1', 'V2']].values.T.tolist()

print(f"The search space contains {len(search_space)} independent variables (the controls the AI can adjust).")
print(f"Topology generated. Number of unique connections (edges): {len(df_topology)}")
print("\nExample of how the computer sees this:")

# Show the first 5 variables in a readable format
for var_name, bounds in list(search_space.items())[:5]:
    print(f"- {var_name}: {bounds}")

print("\nGenerated edge_index (first 5 connections):")
print(f"Source nodes (V1): {edge_index[0][:5]}")
print(f"Target nodes (V2): {edge_index[1][:5]}")

# Save the dictionaries as JSON files
json_search_space_path = config.DATA_IO_PATH / f"search_space_{c11_params.GRID}.json"
json_edge_index_path = config.DATA_IO_PATH / f"edge_index_{c11_params.GRID}.json"
with open(json_search_space_path, 'w') as f:
    json.dump(search_space, f, indent=4) # indent=4 makes it easy to read
with open(json_edge_index_path, 'w') as f:
    json.dump(edge_index, f)

print(f"Search space successfully saved as '{json_search_space_path}'")
print(f"\nEdge index (as a plain list) successfully saved to '{json_edge_index_path}'.")

## 1.6 Representative member length
Samples a subset of generated structures to estimate the representative beam length. This informs timber stock generation (notebook c13) by indicating the length range the stock pool should cover.

In [ ]:
import json
import config
import pandas as pd
import c00_headquarter_params as c11_params
import c12_reconstruction

# Load nodes CSV and edge index produced by section 1.2
nodes_path = config.GEOM_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_nodes.csv"
with open(config.DATA_IO_PATH / f"edge_index_{c11_params.GRID}.json", 'r', encoding='utf-8') as f:
    edge_index = json.load(f)

final_vertices = pd.read_csv(nodes_path)

result = c12_reconstruction.generate_material_statistics(
    df_vertices  = final_vertices,
    edge_index   = edge_index,
    sample_count = 10,
    random_state = 42,
)

summary = result["summary_statistics"]

print("Representative statistics:")
print(f"  Representative length: {result['representative_length_m']:.3f}m ({result['representative_length_mm']:.1f}mm)")
print(f"  Average length: {summary['average_length_m']:.3f}m ({summary['average_length_mm']:.1f}mm)")
print(f"  Median length:  {summary['median_length_m']:.3f}m ({summary['median_length_mm']:.1f}mm)")
print(f"  Min length:     {summary['min_length_m']:.3f}m ({summary['min_length_mm']:.1f}mm)")
print(f"  Max length:     {summary['max_length_m']:.3f}m ({summary['max_length_mm']:.1f}mm)")
print(f"  Total length:   {summary['total_length_m']:.3f}m ({summary['total_length_mm']:.1f}mm)")
print(f"  Std dev:        {summary['std_dev_m']:.3f}m ({summary['std_dev_mm']:.1f}mm)")
print(f"  Q1: {summary['q1_m']:.3f}m  Q3: {summary['q3_m']:.3f}m")
print(f"  Average edge count: {summary['edge_count']}")
print(f"\nSelected samples: {result['selected_sample_ids']}")

# Save to JSON
with open(config.DATA_IO_PATH / "representative_beam_statistics.json", 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2)